# Решения: практика apply и join

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден — положите slim CSV рядом с ноутбуком')


ORDERS_PATH = _find('orders_slim.csv')
CUSTOMERS_PATH = _find('customers_slim.csv')
PAYMENTS_PATH = _find('payments_slim.csv')

orders = pd.read_csv(ORDERS_PATH, parse_dates=['order_purchase_timestamp'])
if 'order_delivered_customer_date' in orders.columns:
    orders['order_delivered_customer_date'] = pd.to_datetime(
        orders['order_delivered_customer_date'], errors='coerce'
    )
customers = pd.read_csv(CUSTOMERS_PATH)
payments = pd.read_csv(PAYMENTS_PATH)


## Урок. 1-5

In [ ]:
orders['order_month'] = orders['order_purchase_timestamp'].dt.month
orders['weekday'] = orders['order_purchase_timestamp'].dt.weekday
payments['is_card'] = (payments['payment_type'] == 'credit_card').astype(int)
joined = orders.merge(payments, on='order_id', how='left')
joined['days_to_deliver'] = joined.apply(
    lambda r: (r['order_delivered_customer_date'] - r['order_purchase_timestamp']).days
    if pd.notna(r['order_delivered_customer_date']) else pd.NA,
    axis=1,
)
p99_delay = float(joined['days_to_deliver'].dropna().quantile(0.99))
n_outliers = int((joined['days_to_deliver'] > p99_delay).sum())
shape_ok = (joined.shape[0] == len(orders) == len(payments))
clean_delay = joined['days_to_deliver'].dropna()
joined.loc[clean_delay.index, 'delay_bin'] = pd.cut(
    clean_delay, bins=[-1, 3, 7, 14, 365], labels=['0-3', '4-7', '8-14', '15+']
)
delay_counts = joined['delay_bin'].value_counts().sort_index()
print(joined.shape, round(p99_delay, 2), n_outliers, shape_ok)
print(delay_counts)

## ДЗ. 1-4

In [ ]:
orders['order_month'] = orders['order_purchase_timestamp'].dt.month
joined = orders.merge(payments, on='order_id', how='left')
month_mean = joined.groupby('order_month')['payment_value'].mean().sort_index()
joined['is_card'] = (joined['payment_type'] == 'credit_card').astype(int)
joined['weekday'] = joined['order_purchase_timestamp'].dt.weekday
weekday_card = joined.groupby('weekday')['is_card'].mean().sort_index()
log_steps = [
    'loaded tables',
    'created order_month and weekday',
    'joined payments to orders',
    'computed days_to_deliver and p99 threshold',
    'binned delays into categories',
]
OUTLIER_NOTE = (
    'Выбросы по задержке полезны как сигнал проблемной логистики, но их нельзя автоматически выбрасывать. '
    'Для части клиентов это реальные кейсы, и они влияют на бизнес-решение, а не только на среднее значение.'
)
print(month_mean.head())
print(weekday_card)
print(log_steps)
print(OUTLIER_NOTE)